# NYC Taxi Trips (11.8M Records) with `vkpandas` + `vkshader`

Terse Datashader-style Vulkan GPU visualization sharing `vknumpy`'s `VkDevice` and `VkBuffer`s with zero host bridging copies (`~15–23 ms` interactive pan & zoom).

In [1]:
import sys, time
from pathlib import Path

_root = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
for _p in (_root, _root.parent / "vkpandas", _root.parent / "delightful-hertz"):
    if _p.is_dir() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import vkpandas as vpd
import vkshader as vks

parquet = Path.home() / ".cache/hvsampledata/nyc_taxi/v2/nyc_taxi_wide.parq"
t0 = time.perf_counter()
df = vpd.read_parquet(parquet, columns=["pickup_x", "pickup_y", "dropoff_x", "dropoff_y"])
print(f"Loaded {len(df):,} trips into GPU vkpandas in {time.perf_counter() - t0:.2f}s")

Loaded 11,842,094 trips into GPU vkpandas in 0.49s


ggml_vulkan: Found 1 Vulkan devices:
ggml_vulkan: 0 = AMD Custom GPU 0405 (RADV VANGOGH) (radv) | uma: 1 | rebar: 1 | fp16: 0 | bf16: 0 | fp4: 0 | warp size: 32 | shared memory: 65536 | int dot: 1 | matrix cores: none


In [2]:
cvs = vks.Canvas(720, 720, x_range=(-8242000, -8226000), y_range=(4968000, 4985000))
vks.interact(
    cvs,
    df,
    "pickup_x",
    "pickup_y",
    how="eq_hist",
    basemap="esri",
    layers={"pickups": ("pickup_x", "pickup_y"), "dropoffs": ("dropoff_x", "dropoff_y")},
);